In [2]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from processing import *
from wavelengths import *
from reprojection import *
from datetime import datetime

In [3]:
q_V = 299792458 / 6173.341
q_B = q_V * 0.231
tuning_constant = 3.513e-4
temperature_constant = 4.01225e-2
alpha = 1.327124e20

In [164]:
files = sorted(glob.glob('/home/ulyanov/data/solo/phi/2026/blos/*.fits'))

In [ ]:
1978
1742

In [287]:
with fits.open(files[1742]) as hdul:
    data1 = hdul[0].data
    header1 = hdul[0].header
    fg_data = hdul['PHI_FITS_FG_settings'].data
    pmp_data = hdul['PHI_FITS_PMP_settings'].data


velocity = header1['OBS_VR']
temperature = header1['FGOV1PT1']
contposn = header1['CONTPOSN']


print(velocity, temperature, contposn)
print(np.nanmedian(data1) / q_V)

10315.1797014435 66.0 blue
5.6208e-07


In [288]:
with fits.open(files[1743]) as hdul:
    data2 = hdul[0].data
    header2 = hdul[0].header
    fg_data = hdul['PHI_FITS_FG_settings'].data
    pmp_data = hdul['PHI_FITS_PMP_settings'].data


velocity = header2['OBS_VR']
temperature = header2['FGOV1PT1']
contposn = header2['CONTPOSN']


print(velocity, temperature, contposn)
print(np.nanmedian(data2) / q_V)

10226.7941767547 61.0 blue
7.9111305e-07


In [289]:
data2 = reproject(data2, header2, header1, correct_mu=True)

data1 = rebin(data1, 8)
data2 = rebin(data2, 8)

t = ~np.isnan(data1) & ~np.isnan(data2)
np.nanmean(np.abs(data1[t])), np.nanmean(np.abs(data2[t]))

(np.float32(3.177792), np.float32(3.1661599))

In [292]:
t = ~np.isnan(data1) & ~np.isnan(data2)

x = data1[t].copy()
y = data2[t].copy()

x -= np.mean(x)
y -= np.mean(y)

A = np.array([[np.mean(x ** 2), np.mean(x * y)],
              [np.mean(x * y), np.mean(y ** 2)]])

vals, vecs = np.linalg.eigh(A)
u, v = vecs[:, 1]

print(v / u)


plt.figure(figsize=(10,10))
plt.plot(x, y, '.', ms=0.5)
plt.plot([-20000,20000], [-20000,20000], 'gray', lw=0.5)

plt.xlim(-300,300)
plt.ylim(-300,300)
plt.tight_layout()

0.9786802


In [286]:
plt.figure(figsize=(10,10))
plt.imshow(data1, 'seismic', vmin=-100, vmax=100)
plt.tight_layout()